# 01 — Data Cleaning

Pipeline de limpieza del dataset crudo **Spanish Wine Quality** (Kaggle).

**Entrada:** `data/raw/dataset/wines_SPA.csv` (7500 filas, 11 columnas)

**Salida:** `data/processed/wines_SPA_clean.csv` — dataset limpio, validado y con las features de negocio, listo para el EDA (`02_eda.ipynb`) y el preprocesamiento (`03_preprocessing.ipynb`).

In [51]:
import pandas as pd
import numpy as np

RAW_PATH = '../data/raw/dataset/wines_SPA.csv'
OUTPUT_PATH = '../data/processed/wines_SPA_clean.csv'
FLAVOR_SCRAPE_PATH = '../data/processed/wines_SPA_enriched.csv'  # salida del scraper (opcional en esta fase)

## 1. Carga del dataset crudo

In [52]:
df = pd.read_csv(RAW_PATH)
print('Forma cruda:', df.shape)
df.head(3)

Forma cruda: (7500, 11)


,winery,wine,year,rating,num_reviews,country,region,price,type,body,acidity
0,Teso La Monja,Tinto,2013,4.9,58,Espana,Toro,995.00,Toro Red,5.0,3.0
1,Artadi,Vina El Pison,2018,4.9,31,Espana,Vino de Espana,313.50,Tempranillo,4.0,2.0
2,Vega Sicilia,Unico,2009,4.8,1793,Espana,Ribera del Duero,324.95,Ribera Del Duero Red,5.0,3.0


## 2. Deduplicación

**Hallazgo clave del EDA inicial:** el 72.7% de las filas del CSV crudo son copias exactas de otra fila (mismo winery, wine, year, rating, price, type... TODAS las columnas idénticas). No es un error nuestro, viene así del CSV de Kaggle. Verificado con un grupo de prueba (`Contino - Reserva - 2016`, repetido 220 veces): las 220 copias tienen exactamente el mismo valor en cada una de las 11 columnas.

Hay que deduplicar **antes** de cualquier otra transformación, porque si no, cualquier estadístico (conteos, medias, el propio scraping) queda inflado por las copias.

In [53]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'{before} filas -> {len(df)} filas únicas ({before - len(df)} duplicados eliminados, '
      f'{(before - len(df)) / before:.1%})')

7500 filas -> 2048 filas únicas (5452 duplicados eliminados, 72.7%)


## 3. Renombrar y quitar columnas que no se usan

In [54]:
df = df.rename(columns={'wine': 'wine_name', 'price': 'price_euros'})

# country siempre vale 'Espana' (no distingue nada); num_reviews, body y acidity
# no forman parte del esquema de features del modelo
df = df.drop(columns=['num_reviews', 'country', 'body', 'acidity'], errors='ignore')

print(df.columns.tolist())

['winery', 'wine_name', 'year', 'rating', 'region', 'price_euros', 'type']


## 4. Separar `type` en `vine_type` (color) y `grape_variety` (uva)

La columna original `type` mezcla color, región y variedad en un mismo string (ej. `'Rioja Red'`, `'Tempranillo'`, `'Ribera Del Duero Red'`). Se separa en dos variables limpias.

In [55]:
WINE_TYPE_KEYWORDS = {
    'Generoso': ['sherry', 'pedro ximenez', 'moscatel', 'muscat', 'muscatel',
                 'amontillado', 'oloroso', 'manzanilla', 'palo cortado', 'cream',
                 'dulce', 'sweet'],
    'Espumoso': ['sparkling', 'cava', 'espumoso', 'frizzante'],
    'Rosado':   ['rose', 'rosado', 'rosé'],
    'Blanco':   ['white', 'blanco', 'albarino', 'verdejo', 'chardonnay',
                 'sauvignon blanc', 'viura', 'godello', 'treixadura', 'palomino',
                 'ribeiro', 'rias baixas'],
    'Tinto':    ['red', 'tinto', 'rouge'],
}

GRAPE_VARIETIES = [
    'Cabernet Sauvignon', 'Sauvignon Blanc', 'Pedro Ximenez', 'Tempranillo',
    'Monastrell', 'Grenache', 'Albarino', 'Verdejo', 'Chardonnay', 'Mencia',
    'Garnacha', 'Syrah', 'Muscatel', 'Muscat', 'Godello', 'Treixadura',
    'Palomino', 'Bobal', 'Viura', 'Macabeo', 'Xarel-lo', 'Parellada',
]

def extract_vine_type(type_str):
    normalized = str(type_str).lower().strip()
    # Orden de prioridad para evitar solapamientos (ej. un generoso dulce
    # también podria matchear 'sweet' antes que 'red')
    for category in ['Generoso', 'Espumoso', 'Rosado', 'Blanco', 'Tinto']:
        if any(kw in normalized for kw in WINE_TYPE_KEYWORDS[category]):
            return category
    return 'Desconocido'

def extract_grape_variety(type_str):
    normalized = str(type_str).lower().strip()
    found = [g for g in GRAPE_VARIETIES if g.lower() in normalized]
    return ' / '.join(found) if found else 'Blend/Other'

df['vine_type'] = df['type'].fillna('').apply(extract_vine_type)
df['grape_variety'] = df['type'].fillna('').apply(extract_grape_variety)
df = df.drop(columns=['type'])

print('vine_type:')
print(df['vine_type'].value_counts())
print()
print(f"grape_variety Blend/Other: {(df['grape_variety']=='Blend/Other').mean():.1%}")

vine_type:
vine_type
Tinto          1529
Desconocido     275
Blanco          115
Generoso         91
Espumoso         38
Name: count, dtype: int64

grape_variety Blend/Other: 86.2%


## 4b. Color por nombre del vino (evidencia directa, antes de inferir por región)

Se comprueba **primero** si el propio nombre del vino ya delata el color (ej. `'Moscatel Blanco Seco'`, `'Godello Blanco'`). Esto tiene que ir antes de las reglas de D.O. del siguiente paso: si no, un vino blanco de una región predominantemente tinta (ej. un Moscatel blanco de Alicante, que es tierra de Monastrell tinto) recibiría por error la uva tinta típica de la región en vez de mantenerse como lo que es.

In [56]:
mask_name_color = df['vine_type'].isin(['Desconocido']) & (
    df['wine_name'].str.contains('Blanco|White', case=False, na=False)
)
df['vine_type_inferred'] = 0
df.loc[mask_name_color, 'vine_type'] = 'Blanco'
df.loc[mask_name_color, 'vine_type_inferred'] = 1

print(f'{mask_name_color.sum()} filas identificadas como Blanco por su propio nombre')

16 filas identificadas como Blanco por su propio nombre


## 5. Completar `grape_variety` con reglas de D.O.

**Hallazgo del EDA:** el matching directo deja ~86-90% de las filas en `'Blend/Other'`, porque `type` casi nunca menciona la uva para vinos regionales genéricos (`'Rioja Red'` no dice Tempranillo explícitamente). Se completa con la variedad predominante de cada D.O., según los consejos reguladores oficiales — **solo** cuando el matching directo no encontró nada, nunca sobrescribiendo un dato ya detectado.

Se marca cada fila completada así en `grape_variety_inferred`, para poder auditar/ponderar distinto el dato inferido del dato directo.

In [57]:
REGION_GRAPE_RULES = [
    # (region, vine_types_aplicables, variedad_asignada)
    ('Rioja',            ['Tinto', 'Desconocido'],           'Tempranillo'),
    ('Rioja',            ['Blanco'],                         'Viura'),
    ('Ribera del Duero', ['Tinto', 'Desconocido'],           'Tempranillo'),
    ('Toro',             ['Tinto', 'Desconocido'],           'Tempranillo'),
    ('Priorato',         ['Tinto', 'Desconocido'],           'Grenache / Carinena'),
    ('Alicante',         ['Tinto', 'Desconocido'],           'Monastrell'),
    ('Ribeiro',          ['Blanco', 'Desconocido'],          'Treixadura'),
    ('El Terrerazo',     ['Tinto', 'Blanco', 'Desconocido'], 'Bobal'),
]

df['grape_variety_inferred'] = 0
for region, vine_types, variety in REGION_GRAPE_RULES:
    mask = (
        (df['region'] == region)
        & (df['vine_type'].isin(vine_types))
        & (df['grape_variety'] == 'Blend/Other')
    )
    df.loc[mask, 'grape_variety'] = variety
    df.loc[mask, 'grape_variety_inferred'] = 1

remaining = (df['grape_variety'] == 'Blend/Other').mean()
print(f"grape_variety completado: {df['grape_variety_inferred'].sum()} filas | "
      f'sigue en Blend/Other: {remaining:.1%} (regiones sin uva dominante clara, ej. Mallorca)')

grape_variety completado: 1364 filas | sigue en Blend/Other: 19.6% (regiones sin uva dominante clara, ej. Mallorca)


## 6. Completar `vine_type` restante usando la variedad de uva

Para lo que sigue en `'Desconocido'` tras el paso 4b, se usa la variedad de uva (directa o recién completada por D.O. en el paso 5) cuando implica el color sin ambigüedad — Tempranillo/Mencía/Monastrell son siempre tintas; Albariño/Verdejo/Godello son siempre blancas.

Filas donde ninguna señal aplica (ej. Mallorca, mezcla real de tintos/rosados/blancos) se dejan en `'Desconocido'` — forzar una regla ahí sería inventar.

In [58]:
RED_VARIETIES = {
    'Tempranillo', 'Monastrell', 'Grenache', 'Grenache / Carinena',
    'Mencia', 'Bobal', 'Syrah', 'Cabernet Sauvignon',
}
WHITE_VARIETIES = {
    'Albarino', 'Verdejo', 'Chardonnay', 'Godello', 'Treixadura',
    'Palomino', 'Viura', 'Macabeo', 'Xarel-lo', 'Parellada', 'Sauvignon Blanc',
}

mask_red = (df['vine_type'] == 'Desconocido') & (df['grape_variety'].isin(RED_VARIETIES))
df.loc[mask_red, 'vine_type'] = 'Tinto'
df.loc[mask_red, 'vine_type_inferred'] = 1

mask_white = (df['vine_type'] == 'Desconocido') & (df['grape_variety'].isin(WHITE_VARIETIES))
df.loc[mask_white, 'vine_type'] = 'Blanco'
df.loc[mask_white, 'vine_type_inferred'] = 1

remaining = (df['vine_type'] == 'Desconocido').mean()
print(f"vine_type completado en total: {df['vine_type_inferred'].sum()} filas | "
      f'sigue en Desconocido: {remaining:.1%}')

vine_type completado en total: 210 filas | sigue en Desconocido: 3.2%


## 7. `wine_ageing` — flag binario de crianza en barrica

Se deriva del **nombre del vino** (no de `type`, que ya se usó para color/uva). `1` = menciona crianza/reserva/barrica; `0` = joven o sin información.

In [59]:
AGEING_POSITIVE_KEYWORDS = [
    'crianza', 'reserva', 'gran reserva', 'roble', 'barrica',
    'aged', 'oak aged', 'barrel', 'madera',
]
AGEING_NEGATIVE_KEYWORDS = ['joven', 'cosechero', 'nuevo', 'young', 'sin crianza']

def is_aged(wine_name):
    text = str(wine_name).lower()
    if any(neg in text for neg in AGEING_NEGATIVE_KEYWORDS):
        return 0  # Explícitamente joven
    if any(pos in text for pos in AGEING_POSITIVE_KEYWORDS):
        return 1  # Crianza confirmada
    return 0      # Sin información -> se asume joven

df['wine_ageing'] = df['wine_name'].fillna('').apply(is_aged)
print(df['wine_ageing'].value_counts())

wine_ageing
0    1624
1     424
Name: count, dtype: int64


## 8. Limpieza de `year`

`'N.V.'` (Non-Vintage, vinos sin añada — típico en generosos/espumosos) se convierte en `NaN` y se imputa con la mediana. El resto de valores no numéricos también caen a `NaN` por seguridad antes de imputar.

In [60]:
df['year'] = df['year'].replace('N.V.', np.nan)
df['year'] = pd.to_numeric(df['year'], errors='coerce')

year_median = df['year'].median()
missing = int(df['year'].isna().sum())
df['year'] = df['year'].fillna(year_median).astype(int)

print(f'year: {missing} valores N.V./no numéricos imputados con la mediana ({int(year_median)})')

year: 72 valores N.V./no numéricos imputados con la mediana (2015)


## 9. Features de negocio

- **`service_temperature`**: temperatura de servicio recomendada, según `vine_type`.
- **`quality_price_ratio`**: `rating / price_euros` — a mayor valor, mejor relación calidad-precio.
- **`luxury_category`**: `1` si `price_euros > 50€`, si no `0` (umbral de negocio, documentado como decisión arbitraria).

In [61]:
SERVING_TEMPERATURE = {
    'Tinto': '16-18°C', 'Blanco': '8-10°C', 'Rosado': '10-12°C',
    'Espumoso': '6-8°C', 'Generoso': '12-14°C', 'Desconocido': '10-14°C',
}
df['service_temperature'] = df['vine_type'].map(SERVING_TEMPERATURE).fillna('10-14°C')

df['quality_price_ratio'] = (df['rating'] / df['price_euros']).round(3)
df['luxury_category'] = (df['price_euros'] > 50).astype(int)

print(df[['service_temperature', 'quality_price_ratio', 'luxury_category']].describe(include='all'))

       service_temperature  quality_price_ratio  luxury_category
count                 2048          2048.000000      2048.000000
unique                   5                  NaN              NaN
top                16-18°C                  NaN              NaN
freq                  1715                  NaN              NaN
mean                   NaN             0.102148         0.519043
std                    NaN             0.086384         0.499759
min                    NaN             0.001000         0.000000
25%                    NaN             0.040000         0.000000
50%                    NaN             0.082000         1.000000
75%                    NaN             0.137000         1.000000
max                    NaN             0.842000         1.000000


## 10. (Opcional) Fusionar `flavor_descriptor` del scraper

El scraping de Vivino (`scrapper/scrapper_enriched.py`) es un proceso aparte (necesita Playwright + conexión, tarda horas). Si ya tienes un CSV con `flavor_descriptor` generado, esta celda lo fusiona por `(winery, wine_name)`. Si no lo tienes todavía, esta celda simplemente se salta — el resto del notebook funciona igual sin sabores, solo faltará esa columna para el modelo final.

In [62]:
import os

if os.path.exists(FLAVOR_SCRAPE_PATH):
    scraped = pd.read_csv(FLAVOR_SCRAPE_PATH)
    flavor_map = scraped.drop_duplicates(subset=['winery', 'wine_name'])[
        ['winery', 'wine_name', 'flavor_descriptor']
    ]
    df = df.merge(flavor_map, on=['winery', 'wine_name'], how='left')
    coverage = df['flavor_descriptor'].notna().mean()
    print(f'flavor_descriptor fusionado. Cobertura: {coverage:.1%}')
else:
    print(f'No se encontró {FLAVOR_SCRAPE_PATH} — se continúa sin flavor_descriptor por ahora.')

flavor_descriptor fusionado. Cobertura: 94.7%


## 11. Deduplicación residual

Al quitar `num_reviews`, `body`, `acidity` (paso 3), pueden aparecer filas que eran técnicamente distintas en el crudo (por esas columnas) pero que ahora, sin ellas, quedan idénticas. Se deduplica una segunda vez tras haber quitado esas columnas.

In [63]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f'{before} -> {len(df)} filas ({before - len(df)} duplicados residuales eliminados)')

2048 -> 2024 filas (24 duplicados residuales eliminados)


## 12. Validación final

Antes de guardar, comprobamos: sin nulos, sin duplicados, y sin inconsistencias `vine_type` / `grape_variety` (ej. un vino marcado Tinto con una uva exclusivamente blanca).

In [64]:
def validate(df):
    issues = []
    n_nulls = df.isna().sum().sum()
    if n_nulls > 0 and 'flavor_descriptor' not in df.columns:
        issues.append(f'{n_nulls} nulos residuales')
    n_dups = df.duplicated().sum()
    if n_dups > 0:
        issues.append(f'{n_dups} filas duplicadas')

    red = {'Tempranillo','Monastrell','Grenache','Grenache / Carinena','Mencia',
           'Bobal','Syrah','Cabernet Sauvignon'}
    white = {'Albarino','Verdejo','Chardonnay','Godello','Treixadura','Palomino',
             'Viura','Macabeo','Xarel-lo','Parellada','Sauvignon Blanc'}
    inconsist = df[
        ((df['vine_type'] == 'Blanco') & (df['grape_variety'].isin(red)))
        | ((df['vine_type'] == 'Tinto') & (df['grape_variety'].isin(white)))
    ]
    if len(inconsist) > 0:
        issues.append(f'{len(inconsist)} filas con vine_type/grape_variety inconsistentes')

    if issues:
        print('Problemas encontrados:')
        for i in issues:
            print(' -', i)
        return inconsist
    else:
        print('Validación OK: sin nulos (excepto flavor_descriptor si aún no se scrapeó), '
              'sin duplicados, sin inconsistencias.')
        return None

inconsistencias = validate(df)
inconsistencias

Validación OK: sin nulos (excepto flavor_descriptor si aún no se scrapeó), sin duplicados, sin inconsistencias.


Si `validate()` encontró alguna inconsistencia puntual, se revisa y corrige a mano aquí antes de guardar (normalmente son 0-1 casos, típicos de reglas de D.O. aplicadas a una región con estilos mixtos).

In [65]:
# Ejemplo de corrección puntual si hiciera falta (ajustar índices/valores según lo que salga arriba):
# df.loc[df['wine_name'] == 'NOMBRE_DEL_CASO', 'grape_variety'] = 'Blend/Other'
# df.loc[df['wine_name'] == 'NOMBRE_DEL_CASO', 'grape_variety_inferred'] = 0

## 13. Quitar columnas de auditoría antes de guardar

`vine_type_inferred` y `grape_variety_inferred` solo sirvieron para verificar durante la limpieza cuántas filas se completaron por regla vs. detectadas directamente (pasos 4b, 5 y 6). No forman parte del esquema final que necesita el modelo — se descartan y se dejan solo `vine_type` y `grape_variety` con su valor final ya resuelto.

In [66]:
df = df.drop(columns=['vine_type_inferred', 'grape_variety_inferred'], errors='ignore')
print('Columnas finales:', df.columns.tolist())

Columnas finales: ['winery', 'wine_name', 'year', 'rating', 'region', 'price_euros', 'vine_type', 'grape_variety', 'wine_ageing', 'service_temperature', 'quality_price_ratio', 'luxury_category', 'flavor_descriptor']


## 14. Guardar resultado

In [67]:
df.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f'Guardado: {OUTPUT_PATH}')
print(f'Forma final: {df.shape}')
print(f'Columnas: {df.columns.tolist()}')

Guardado: ../data/processed/wines_SPA_clean.csv
Forma final: (2024, 13)
Columnas: ['winery', 'wine_name', 'year', 'rating', 'region', 'price_euros', 'vine_type', 'grape_variety', 'wine_ageing', 'service_temperature', 'quality_price_ratio', 'luxury_category', 'flavor_descriptor']
